# Create an upload job for dataset in TDEI system
- Get the dataset detail from TDEI API
- Create an updated dataset metadata from the response from API
- Login to MC orchestrator
- Make a new upload request for the dataset with Metadata, accesstoken and other details
- Check the upload request and confirm upload

In [375]:
from dotenv import load_dotenv
import os
load_dotenv()
tdei_username = os.environ.get('TDEI_USERNAME')
tdei_password = os.environ.get('TDEI_PASSWORD')
base_url = 'https://api.tdei.us'
datasets_path = '/api/v1/datasets'
auth_path = '/api/v1/authenticate'


In [376]:
import requests

url = base_url + auth_path
payload = {
    'username': tdei_username,
    'password': tdei_password
}
headers = {
    'Content-Type': 'application/json'
}

response = requests.post(url, json=payload, headers=headers)
response.json()
access_token = response.json()['access_token']

In [377]:
mc_base_url = 'https://wa-proviso-api-dev.azurewebsites.net'
login_path = '/auth/login'
mc_user_name = os.environ.get('MC_USERNAME')
mc_password = os.environ.get('MC_PASSWORD')

# Login to the MC orchestrator


url = mc_base_url + login_path
payload = {
    'email': mc_user_name,
    'password': mc_password
}
headers = {
    'Content-Type': 'application/json'
}

response = requests.post(url, json=payload, headers=headers)

mc_access_token = response.json()['access_token']


In [402]:
tdei_dataset_id = 'cc43a93d-3808-487d-98bb-90758c76c0bf'
mc_project_id = '677e398ccc47ef650955a86b'

In [403]:
url = f'https://api.tdei.us/api/v1/datasets?'
url += 'page_no=1&page_size=10&sort_field=uploaded_timestamp&sort_order=DESC&status=All&'
url += f'tdei_dataset_id={tdei_dataset_id}&'
url += 'tdei_project_group_id=1dd7c38e-c7a6-4e3a-be8b-379f823a7ad7'
response = requests.get(url, headers={'Authorization': 'Bearer ' + access_token})
data = response.json()
if len(data) > 0:
    dataset = data[0]
    project_group = dataset['project_group']['tdei_project_group_id']
    service_id = dataset['service']['tdei_service_id']
    metadata = dataset['metadata']
else:
    print('Dataset not found')

existing_version = metadata['dataset_detail']['version']    
name = metadata['dataset_detail']['name']

def get_new_version(old_version):
    # split the version into major, minor
    major, minor = old_version.split('.')
    # increment the patch version
    minor = str(int(minor) + 1)
    # If minor is more than 9, increment the major version
    if int(minor) > 9:
        major = str(int(major) + 1)
        minor = '0'
    # return the new version
    return f'{major}.{minor}'

new_version = get_new_version(existing_version)
print(name)
print(f'Old version {existing_version}, New version {new_version}')
    
boundary = metadata['dataset_detail']['dataset_area']
# Get the feature[0] as boundary
if boundary['features']:
    boundary = boundary['features'][0]
else:
    print(f'No boundary available.')
release_notes = metadata['dataset_summary']['release_notes']
release_notes += f'Hxgn fixes added'


GS_Bucoda_City
Old version 1.7, New version 1.8


In [404]:
new_metadata = metadata.copy()
new_metadata['dataset_summary']['release_notes'] += f'Hxgn fixes added\n'
new_metadata['dataset_detail']['version'] = new_version
# new_metadata['dataset_detail']['dataset_area'] = boundary
if not boundary['properties']:
    boundary['properties'] = {}
boundary['properties']['name'] = ''
boundary['properties']['id'] = mc_project_id
boundary['properties']['tdei_service_id'] = service_id
boundary['properties']['tdei_pg_id'] = project_group


In [405]:
dataset_upload_api = f'/projects/{mc_project_id}/upload-tdei-dataset'


In [406]:
mc_upload_request_url = mc_base_url+dataset_upload_api

mc_tdei_upload_payload = {
    'boundary':boundary,
    'metadata': new_metadata,
    'access_token': access_token,
    'environment' : 'prod'
}
request_headers = {
     'Content-Type': 'application/json',
     'Authorization': 'Bearer '+mc_access_token
}
upload_request_response = requests.post(mc_upload_request_url,json=mc_tdei_upload_payload, headers= request_headers)
print(upload_request_response.json())

{'message': 'Flow generated successfully', 'flow_id': 'a7490a75-18ea-4949-b8d3-063f369001f9'}
